### Set up

In [26]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Load the data
simple_train_data = pd.read_csv('data/simple-train.csv')
simple_test_data = pd.read_csv('data/simple-test.csv')
netflix_train_data = pd.read_csv('data/netflix-train.csv')
netflix_test_data = pd.read_csv('data/netflix-test.csv')
netflix_train_data_no_demo = pd.read_csv('data/netflix-train.csv', usecols=lambda x: x not in ['Demographic'])
netflix_test_data_no_demo = pd.read_csv('data/netflix-test.csv', usecols=lambda x: x not in ['Demographic'])
ancestry_train_data = pd.read_csv('data/ancestry-train.csv')
ancestry_test_data = pd.read_csv('data/ancestry-test.csv')
heart_train_data = pd.read_csv('data/heart-train.csv')
heart_test_data = pd.read_csv('data/heart-test.csv')
simple_train_data.head()


,x1,x2,Label
0,0,0,0
1,0,1,0
2,1,0,1
3,1,1,1


### Implement the Naive Bayes Classifier

##### Compute the prior probability of the target variable Y

In [27]:
def compute_prior(data):
    """
    Compute the prior probability of the target variable Y taking on value 1.

    Args:
        data (pd.DataFrame): The training data.

    Returns:
        The prior probability of the target variable Y taking on value 1.
    """
    return data['Label'].sum() / len(data)
compute_prior(simple_train_data)


np.float64(0.5)

##### Compute the conditional probability of each feature

In [28]:
def compute_feature_MLE(data):
    """
    Compute the conditional probability of the feature X_i taking on value 1 given that Y = 1 
    using Maximum Likelihood Estimators (MLE).

    Args:
        data (pd.DataFrame): The training data.

    Returns:
        The conditional probability of the feature X_i taking on value 1 given that Y = 0 or Y = 1.
    """
    # Stros the probability of each feature taking on value 1 when Y = 0 and Y = 1 seperatly
    p0 = {}
    p1 = {}
    # Compute the probability of each feature
    for col in data.iloc[:, :data.shape[1] - 1]:
        p0[col] = data[col][data['Label'] == 0].sum() / len(data[data['Label'] == 0])
        p1[col] = data[col][data['Label'] == 1].sum() / len(data[data['Label'] == 1])
    # Return the probability of each feature
    return pd.Series(p0), pd.Series(p1)
compute_feature_MLE(simple_train_data)

(x1    0.0
 x2    0.5
 dtype: float64,
 x1    1.0
 x2    0.5
 dtype: float64)

In [29]:
def compute_feature_LAP(data):
    """
    Compute the conditional probability of the feature X_i taking on value 1 given that Y = 1
    using Laplace Estimators.

    Args:
        data (pd.DataFrame): The training data.

    Returns:
        The conditional probability of the feature X_i taking on value 1 given that Y = 0 or Y = 1.
    """
    # Stros the probability of each feature taking on value 1 when Y = 0 and Y = 1 seperatly
    p0 = {}
    p1 = {}
    # Compute the probability of each feature
    for col in data.iloc[:, :data.shape[1] - 1]:
        p0[col] = (data[col][data['Label'] == 0].sum() + 1) / (len(data[data['Label'] == 0]) + 2)
        p1[col] = (data[col][data['Label'] == 1].sum() + 1) / (len(data[data['Label'] == 1]) + 2)
    # Return the probability of each feature
    return pd.Series(p0), pd.Series(p1)
compute_feature_LAP(simple_train_data)

(x1    0.25
 x2    0.50
 dtype: float64,
 x1    0.75
 x2    0.50
 dtype: float64)

##### Compute the likelihood of Y = 1 given the value of features

In [30]:
def compare_likelihood(test_data, train_data, p0, p1, p_Y):
    """
    Compute and compare the likelihood of the features given the prior. Returns 1 if Y = 1 
    has the higher likelihood.

    Args:
        data (pd.DataFrame): The test data.
        p0 (pd.Series): The conditional probability of the features taking on value 1 when Y = 0.
        p1 (pd.Series): The conditional probability of the features taking on value 1 when Y = 1.
        p_Y (float): The prior probability of the target variable Y taking on value 1.

    Returns:
        The predictions of Y for each row of the testing data
    """
    prediction = {}
    # Compare the likelihood given Y = 0 and 1 for each row
    for index, row in test_data.iterrows():
        # Compute the likelihood given the prior is 0
        likelihood0 = 1
        for col in train_data.columns[:-1]:
            value = row[col]
            p = p0[col] if value == 1 else 1 - p0[col]
            likelihood0 *= p
        likelihood0 *= (1 - p_Y)
        # Compute the likelihood given the prior is 1
        likelihood1 = 1
        for col in train_data.columns[:-1]:
            value = row[col]
            p = p1[col] if value == 1 else 1 - p1[col]
            likelihood1 *= p
        likelihood1 *= p_Y
        # Compare the likelihood and return 1 if Y = 1 has the higher likelihood
        prediction[index] = 1 if likelihood1 > likelihood0 else 0
    return pd.Series(prediction)


##### Construct the Naive Bayes Classifier

In [31]:
def naive_bayes_MLE(train_data, test_data):
    """
    Pridict the value of the label using Maximum Likelihood Estimators (MLE)

    Args:
        train_data (pd.DataFrame): The training data.
        test_data (pd.DataFrame): The test data.

    Returns:
        likelihood (pd.Series): The pridicted value of the label
    """
    # Compute the prior probability
    p_Y = compute_prior(train_data)
    # Compute the conditional probability
    p0, p1 = compute_feature_MLE(train_data)
    # Pridict the prior based on the traing data
    prediction = compare_likelihood(test_data, train_data, p0, p1, p_Y)
    return prediction
naive_bayes_MLE(simple_train_data, simple_test_data)

0    0
1    0
2    1
3    1
dtype: int64

In [32]:
def naive_bayes_LAP(train_data, test_data):
    """
    Pridict the value of the label using Laplace Estimators

    Args:
        train_data (pd.DataFrame): The training data.
        test_data (pd.DataFrame): The test data.

    Returns:
        likelihood (pd.Series): The pridicted value of the label
    """
    # Compute the prior probability
    p_Y = compute_prior(train_data)
    # Compute the conditional probability
    p0, p1 = compute_feature_LAP(train_data)
    # Pridict the prior based on the traing data
    prediction = compare_likelihood(test_data, train_data, p0, p1, p_Y)
    return prediction
naive_bayes_LAP(simple_train_data, simple_test_data)

0    0
1    0
2    1
3    1
dtype: int64

### Check accuracy

In [33]:
def check_accuracy(test_data, prediction):
    """
    Check the accuracy of the prediction and format the printed message

    Args:
        test_data (pd.DataFrame): The test data
        prediction (pd.Series): The prediction for the test data generated by the classifier
    """
    tested0 = (test_data['Label'] == 0).sum()
    tested1 = (test_data['Label'] == 1).sum()
    testedAll = len(test_data)
    correct0 = ((test_data['Label'] == 0) & (prediction == 0)).sum()
    correct1 = ((test_data['Label'] == 1) & (prediction == 1)).sum()
    correctAll = (test_data['Label'] == prediction).sum()
    accuracy = (correct0 + correct1) / testedAll
    print(f'''
Class 0: tested {tested0}, correctly classified {correct0}
Class 1: tested {tested1}, correctly classified {correct1}
Overall: tested {testedAll}, correctly classified {correctAll}
Accuracy: {accuracy}
''')

check_accuracy(simple_test_data, naive_bayes_LAP(simple_train_data, simple_test_data))


Class 0: tested 2, correctly classified 2
Class 1: tested 2, correctly classified 2
Overall: tested 4, correctly classified 4
Accuracy: 1.0



### a. Naive Bayes Classifier for simple datas

In [34]:
check_accuracy(simple_test_data, naive_bayes_MLE(simple_test_data, simple_train_data))


Class 0: tested 2, correctly classified 2
Class 1: tested 2, correctly classified 2
Overall: tested 4, correctly classified 4
Accuracy: 1.0



In [35]:
check_accuracy(simple_test_data, naive_bayes_LAP(simple_test_data, simple_train_data))


Class 0: tested 2, correctly classified 2
Class 1: tested 2, correctly classified 2
Overall: tested 4, correctly classified 4
Accuracy: 1.0



### b. Naive Bayes Classifier for Netflix datas

##### (1) P(Y = 1)

In [36]:
compute_prior(netflix_train_data_no_demo)

np.float64(0.5186666666666667)

##### (2) P(X_i = 1 | Y = 1)

In [37]:
p0, p1 = compute_feature_MLE(netflix_train_data_no_demo)
p1.head()

3 Idiots                        0.717652
Bourne Identity                 0.655955
Bruce Almighty                  0.603685
Forest Gump                     0.616538
How to Lose a Guy in 10 Days    0.730077
dtype: float64

##### (3)P(X_i = 1 | Y = 0)

In [38]:
p0.head()

3 Idiots                        0.623269
Bourne Identity                 0.669437
Bruce Almighty                  0.659741
Forest Gump                     0.662050
How to Lose a Guy in 10 Days    0.591874
dtype: float64

##### (4) Report the accuracy

In [39]:
check_accuracy(netflix_test_data_no_demo, naive_bayes_MLE(netflix_train_data_no_demo, netflix_test_data_no_demo))


Class 0: tested 248, correctly classified 155
Class 1: tested 252, correctly classified 181
Overall: tested 500, correctly classified 336
Accuracy: 0.672



In [40]:
check_accuracy(netflix_test_data_no_demo, naive_bayes_LAP(netflix_train_data_no_demo, netflix_test_data_no_demo))


Class 0: tested 248, correctly classified 155
Class 1: tested 252, correctly classified 181
Overall: tested 500, correctly classified 336
Accuracy: 0.672



### c. Analyze the Netflix dataset

##### (1) Find the 5 most indicative movies

In [41]:
def compute_p_Y_1(data):
    """
    Compute the probability of Y = 1 when the feature takes on value 1 for each feature.
    
    Args:
        data (pd.DataFrame): The training data.

    Returns:
        The conditional probability of Y = 1 when the feature takes on value 0 or 1
    """
    # Stros the probability for each feature
    p0 = {}
    p1 = {}
    # Compute the probability of each feature
    for col in data.columns[:-1]:
        p0[col] = data['Label'][data[col] == 0].sum() / len(data[data[col] == 0])
        p1[col] = data['Label'][data[col] == 1].sum() / len(data[data[col] == 1])
    return pd.Series(p0), pd.Series(p1)
compute_p_Y_1(simple_train_data)
        

(x1    0.0
 x2    0.5
 dtype: float64,
 x1    1.0
 x2    0.5
 dtype: float64)

In [42]:
def compute_ratio(p0, p1):
    """
    Returns the ratio of P(Y = 1 | X_i = 1) and P(Y = 1 | X_i = 0)

    Args:
        p0 (pd.Series): P(Y = 1 | X_i = 0) for each X_i
        p1 (pd.Series): P(Y = 1 | X_i = 1) for each X_i

    Returns:
        ratio (pd.Series): P(Y = 1 | X_i = 1) / P(Y = 1 | X_i = 0) in descending order
    """
    ratio = p0 / p1
    return ratio.sort_values(ascending=False)

In [43]:
# Compute the movies with the 5 highest ratio
p0, p1 = compute_p_Y_1(netflix_train_data_no_demo)
compute_ratio(p0, p1).head()

Bruce Almighty    1.121017
Forest Gump       1.098273
Rat Race          1.088516
Shrek             1.065731
Oceans 11         1.053591
dtype: float64

##### (2) Find one miss-prediction and explain the reason

In [44]:
def find_incorrect(test_data, prediction):
    """
    Find the rows with the incorrect pridiction

    Args:
        test_data (pd.DataFrame): The test data
        prediction (pd.Series): The prediction for the test data generated by the classifier

    Returns:
        incorrect_rows (pd.Series): The number of incorrect rows
    """
    incorrect_rows = (test_data['Label'] != prediction)
    return incorrect_rows[incorrect_rows == True]
find_incorrect(simple_test_data, naive_bayes_MLE(simple_train_data, simple_test_data))

Series([], dtype: bool)

In [45]:
# Find the rows with incorrect predictions
find_incorrect(netflix_test_data_no_demo, naive_bayes_MLE(netflix_train_data_no_demo, netflix_test_data_no_demo))

12     True
17     True
19     True
21     True
24     True
       ... 
486    True
495    True
496    True
498    True
499    True
Length: 164, dtype: bool

In [46]:
# Print the values of the first incorrectly-pridicted row
netflix_test_data_no_demo.iloc[12]

3 Idiots                        1
Bourne Identity                 1
Bruce Almighty                  0
Forest Gump                     0
How to Lose a Guy in 10 Days    0
I Robot                         0
Independence Day                0
La Vita E Bella                 1
Lord of the Rings               1
Oceans 11                       1
Patriot                         1
Pearl Harbor                    0
Pirates                         1
Pulp Fiction                    1
Rat Race                        0
Shrek                           0
Star Wars                       0
What Women Want                 0
When Harry Met Sally            0
Label                           1
Name: 12, dtype: int64

Explain the misprediction:

In row 12 of the test data, 4 of the 5 of the most influencial movies is not liked (of value 0), which leads to the pridiction is alargely based on the less influential movies, and which increases the chance of misprsdiction

### d. Naive Bayes Classifier for ancestry datas

In [47]:
check_accuracy(ancestry_test_data, naive_bayes_MLE(ancestry_train_data, ancestry_test_data))


Class 0: tested 109, correctly classified 95
Class 1: tested 75, correctly classified 50
Overall: tested 184, correctly classified 145
Accuracy: 0.7880434782608695



In [48]:
check_accuracy(ancestry_test_data, naive_bayes_LAP(ancestry_train_data, ancestry_test_data))


Class 0: tested 109, correctly classified 94
Class 1: tested 75, correctly classified 49
Overall: tested 184, correctly classified 143
Accuracy: 0.7771739130434783



### d. Naive Bayes Classifier for heart datas

In [49]:
check_accuracy(heart_test_data, naive_bayes_MLE(heart_train_data, heart_test_data))


Class 0: tested 15, correctly classified 10
Class 1: tested 172, correctly classified 138
Overall: tested 187, correctly classified 148
Accuracy: 0.7914438502673797



In [50]:
check_accuracy(heart_test_data, naive_bayes_LAP(heart_train_data, heart_test_data))


Class 0: tested 15, correctly classified 10
Class 1: tested 172, correctly classified 131
Overall: tested 187, correctly classified 141
Accuracy: 0.7540106951871658



### f. The code copy (see the code above)